In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


# Load and Inspect Data

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv")

First columns of data

In [ ]:
df.head()

Last columns of dataset

In [ ]:
df.tail()

Basic math calcualtion

In [ ]:
df.describe()

See rows and column numbers of dataset

In [ ]:
df.shape

# Finding Null Values

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

**Plot**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.heatmap(df.isnull(), cbar = False , cmap = "viridis")
plt.title("Missing value heatmap")
plt.show()

In [ ]:
df.info()

# Encode Categorical columns

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

In [ ]:
for i in df:
    if(df[i].dtypes == 'object'):
        df[i] = le.fit_transform(df[i])
        print(i,df[i].dtypes )

In [ ]:
for i in df:
    print(df[i].value_counts())

# Train/Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop("math score", axis=1)
y = df["math score"]
len(X)

In [ ]:
len(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=50)

In [ ]:
print('Size of Train X', len(X_train))
print('Size of Train y',len(y_train))
print('Size of Test X',len(X_test))
print('Size of Test y',len(y_test))

In [ ]:
from sklearn.ensemble import RandomForestRegressor


***Why i choose this model?***

* It handles non-linear relationships effectively, which is common in student performance data (e.g., interaction between study habits and scores).
* It reduces overfitting compared to a single decision tree by averaging multiple trees.
* It works well on medium-sized tabular datasets, such as exam performance datasets.

In [ ]:
regressor = RandomForestRegressor(n_estimators = 100,  random_state = 32)
regressor.fit(X_train, y_train)

In [ ]:
training_data_prediction_RF = regressor.predict(X_train)
print(training_data_prediction_RF)

In [ ]:
from sklearn.metrics import(
mean_absolute_error, mean_squared_error, r2_score,
mean_absolute_percentage_error
)

In [ ]:
Train_r2_RF = r2_score(y_train, training_data_prediction_RF)
Train_mse_RF = mean_squared_error (y_train, training_data_prediction_RF)
Train_rmse_RF = np.sqrt(Train_mse_RF)
print(f"R2 : {Train_r2_RF : .4f}")
print(f"MSE : {Train_mse_RF : .4f}")
print(f"RMSE : {Train_rmse_RF : .4f}")

In [ ]:
y_predict_RF = regressor.predict(X_test)
y_predict_RF[: 5]

In [ ]:
Test_r2_RF = r2_score(y_test, y_predict_RF)
Test_mse_RF = mean_squared_error (y_test, y_predict_RF)
Test_rmse_RF = np.sqrt(Test_mse_RF)
print(f"R2 : {Test_r2_RF : .4f}")
print(f"MSE : {Test_mse_RF : .4f}")
print(f"RMSE : {Test_rmse_RF : .4f}")

***Why i choose XGBoost  Model?***

1. It builds models sequentially, correcting errors from previous trees, leading to high accuracy.
2. It includes regularization (L1 & L2), which helps prevent overfitting.
3. It efficiently handles complex feature interactions.
4. It is widely used in competitive ML and structured/tabular data problems, often outperforming other regression models.

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=32
)

xgb.fit(X_train, y_train)

In [ ]:
training_data_prediction_XGB = xgb.predict(X_train)
print(training_data_prediction_XGB)

In [ ]:
Train_r2_Xgb = r2_score(y_train, training_data_prediction_XGB)
Train_mse_Xgb = mean_squared_error (y_train, training_data_prediction_XGB)
Train_rmse_Xgb = np.sqrt(Train_mse_Xgb)
print(f"R2 : {Train_r2_Xgb : .4f}")
print(f"MSE : {Train_mse_Xgb : .4f}")
print(f"RMSE : {Train_rmse_Xgb : .4f}")

In [ ]:
y_pred_Xgb = xgb.predict(X_test)
y_pred_Xgb[: 5]

In [ ]:
Test_r2_Xgb = r2_score(y_test, y_pred_Xgb)
Test_mse_Xgb = mean_squared_error (y_test, y_pred_Xgb)
Test_rmse_Xgb = np.sqrt(Test_mse_Xgb)
print(f"R2 : {Test_r2_Xgb : .4f}")
print(f"MSE : {Test_mse_Xgb : .4f}")
print(f"RMSE : {Test_rmse_Xgb : .4f}")

# Comparison

In [ ]:
results = {
    
    "Random Forest": {
        "Train R2": Train_r2_RF,
        "Test R2": Test_r2_RF,
        "Train RMSE": Train_rmse_RF,
        "Test RMSE": Test_rmse_RF
    },

    "XGBoost": {
        "Train R2": Train_r2_Xgb,
        "Test R2": Test_r2_Xgb,
        "Train RMSE": Train_rmse_Xgb,
        "Test RMSE": Test_rmse_Xgb
    },

}

Sort models based on thier testing score of r2.

In [ ]:
results_df.sort_values(by='Test R2', ascending=False)

**Feature engineering for classification**

**The pass_fail column is created using:**
* math score >= 50 → 1 (Pass)
* math score < 50 → 0 (Fail)

In [ ]:
df['pass_fail'] = (df['math score'] >= 50).astype(int)

***Why it is created***

1. The original dataset contains continuous exam scores, which are suitable for regression analysis.
2. However, many real-world educational decisions are binary outcomes, such as:
 * Pass / Fail
 * Qualified / Not Qualified
 * Eligible / Not Eligible
3. Therefore, converting scores into pass_fail enables:
   * Binary classification modeling
   * Easier interpretation of student performance outcomes
   * Alignment with real-world academic decision-making systems  

In [ ]:
df['pass_fail'].value_counts()

# Classification models

In [ ]:
df.info()

***Why it is used as target variable***

*pass_fail is selected as the target variable (y) because:*
* It represents the final outcome of interest (student success/failure)
* It is derived from academic performance metric (math score) using a defined threshold
* It allows supervised learning models to learn patterns that distinguish:
     * High-performing students (Pass)
     * Low-performing students (Fail)

In [ ]:
X = df.drop("pass_fail", axis=1)
y = df["pass_fail"]
len(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=50)

**Logistic Regression**

***Why Logistic Regression for Classification?***

*Logistic Regression is selected because:*
* It is a baseline linear classification model used for comparison.
* It is highly interpretable, allowing understanding of feature impact on probability of passing.
* It works well when the relationship between variables and outcome is approximately linear.
* It provides probabilistic outputs, useful for classification evaluation.
* It is computationally efficient and widely used in educational analytics research.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
LModel = LogisticRegression(solver='liblinear',random_state=10)

In [ ]:
LModel.fit(X_train,y_train)
LogisticRegression(random_state=10, solver='liblinear')

In [ ]:
training_data_prediction_lr = LModel.predict(X_train)
print(training_data_prediction_lr)

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [ ]:
print("Accuracy : ",accuracy_score(y_train,training_data_prediction_lr))
print("Precission : ",precision_score(y_train,training_data_prediction_lr,average='weighted'))
print("Recall_score : ",recall_score(y_train,training_data_prediction_lr,average='weighted'))
print("F1_score : ",f1_score(y_train,training_data_prediction_lr,average='weighted'))

In [ ]:
testing_data_prediction_lr = LModel.predict(X_test)
print(testing_data_prediction_lr)

In [ ]:
print("Accuracy : ",accuracy_score(y_test,testing_data_prediction_lr))
print("Precission : ",precision_score(y_test,testing_data_prediction_lr,average='weighted'))
print("Recall_score : ",recall_score(y_test,testing_data_prediction_lr,average='weighted'))
print("F1_score : ",f1_score(y_test,testing_data_prediction_lr,average='weighted'))

Classification report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, testing_data_prediction_lr))

Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
cm = confusion_matrix(y_train, training_data_prediction_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

**Random Forest Classifier**

**Why Random Forest**
*Random Forest is chosen because:*
* It handles non-linear decision boundaries effectively.
* It is robust to noise and outliers in educational datasets.
* It provides feature importance, helping interpret which factors influence pass/fail.
* It reduces overfitting through ensemble averaging.
* It performs well without heavy preprocessing.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model =RandomForestClassifier(n_estimators=10, random_state=15)
rf_model.fit(X_train, y_train)

In [ ]:
training_data_prediction_rf =rf_model.predict(X_train)
print(training_data_prediction_rf)

In [ ]:
print("Accuracy : ",accuracy_score(y_train,training_data_prediction_rf))
print("Precission : ",precision_score(y_train,training_data_prediction_rf,average='weighted'))
print("Recall_score : ",recall_score(y_train,training_data_prediction_rf,average='weighted'))
print("F1_score : ",f1_score(y_train,training_data_prediction_rf,average='weighted'))

In [ ]:
testing_data_prediction_rf = rf_model.predict(X_test)
print(testing_data_prediction_rf)

In [ ]:
print("Accuracy : ",accuracy_score(y_test,testing_data_prediction_rf))
print("Precission : ",precision_score(y_test,testing_data_prediction_rf,average='weighted'))
print("Recall_score : ",recall_score(y_test,testing_data_prediction_rf,average='weighted'))
print("F1_score : ",f1_score(y_test,testing_data_prediction_rf,average='weighted'))

Classification Report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, testing_data_prediction_rf))

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_train, training_data_prediction_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()